# 01 — Well Positioning

Solve electrode voltages that place an approximate harmonic potential well at selected axial target positions.

This notebook follows `00_trap_basis.ipynb`:

```text
electrode basis → target well → voltage solution → positioned potential
```

The goal is not full device fidelity yet. The goal is a transparent first-pass control layer that can later be extended toward segmented RF Paul trap waveform design.


## 1. Imports and path setup

For GitHub/Colab, this assumes the repo root is the current working directory. If running locally from `notebooks/`, use the small path fallback below.


In [ ]:
from pathlib import Path
import sys

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import numpy as np
import matplotlib.pyplot as plt

from src.ion_transport_waveform.config import TrapConfig
from src.ion_transport_waveform.trap_model import (
    gaussian_electrode_basis,
    potential_from_voltages,
    electric_field,
    curvature_at_target,
)
from src.ion_transport_waveform.waveform_solver import solve_voltages_for_target_well

fig_dir = repo / "figures"
fig_dir.mkdir(exist_ok=True)


## 2. Build segmented-electrode basis

We reuse the simplified 1D Gaussian electrode-basis model from Notebook 00. Each electrode contributes a smooth local basis potential, and electrode voltages combine linearly.


In [ ]:
cfg = TrapConfig()

x = np.linspace(-420e-6, 420e-6, 1400)
electrode_positions = np.arange(-5, 6) * cfg.electrode_pitch_m
basis = gaussian_electrode_basis(x, electrode_positions, cfg.basis_width_m)

print(f"x grid: {x.size} points")
print(f"electrodes: {len(electrode_positions)}")
print(f"voltage limit: ±{cfg.voltage_limit_v:.1f} V")


## 3. Solve voltages for several target well positions

For each target position, solve a ridge-regularized least-squares problem:

```text
basis @ voltages ≈ target harmonic well
```

This is intentionally modest and readable: it provides the control-synthesis scaffold before full trap-field modeling.


In [ ]:
targets_um = np.array([-160, -80, 0, 80, 160], dtype=float)
targets = targets_um * 1e-6

solutions = []
for target_x in targets:
    v = solve_voltages_for_target_well(
        basis=basis,
        x_grid=x,
        target_x=target_x,
        voltage_limit=cfg.voltage_limit_v,
        ridge=1e-5,
    )
    phi = potential_from_voltages(basis, v)
    E = electric_field(x, phi)
    kappa = curvature_at_target(x, phi, target_x)
    min_x = x[np.argmin(phi)]
    solutions.append({
        "target_x": target_x,
        "voltages": v,
        "potential": phi,
        "field": E,
        "curvature": kappa,
        "min_x": min_x,
    })

for s in solutions:
    print(
        f"target={s['target_x']*1e6:7.1f} µm | "
        f"min={s['min_x']*1e6:7.1f} µm | "
        f"max|V|={np.max(np.abs(s['voltages'])):5.2f} V | "
        f"curvature={s['curvature']:.3e} arb/m²"
    )


## 4. Figure: positioned wells

This figure is the first paper/site result for the control layer: voltage solutions can move the approximate well center across the trap axis.


In [ ]:
plt.figure(figsize=(8.2, 4.8))

for s in solutions:
    phi = s["potential"]
    phi_norm = (phi - np.min(phi)) / (np.max(phi) - np.min(phi) + 1e-12)
    plt.plot(x * 1e6, phi_norm, label=f"target {s['target_x']*1e6:.0f} µm")
    plt.axvline(s["target_x"] * 1e6, linestyle="--", linewidth=0.8, alpha=0.35)

plt.xlabel("axial position (µm)")
plt.ylabel("normalized potential (arb.)")
plt.title("Voltage-solved approximate well positions")
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig(fig_dir / "well_positioning.png", dpi=180)
plt.show()

print(f"saved: {fig_dir / 'well_positioning.png'}")


## 5. Figure: electrode voltage patterns

The voltage patterns show how the solution changes as the target well moves. This will become useful in Notebook 03 when each target position becomes one timestep in a waveform.


In [ ]:
plt.figure(figsize=(8.2, 4.6))

for s in solutions:
    plt.plot(
        electrode_positions * 1e6,
        s["voltages"],
        marker="o",
        label=f"target {s['target_x']*1e6:.0f} µm",
    )

plt.axhline(cfg.voltage_limit_v, linestyle="--", linewidth=0.8, alpha=0.4)
plt.axhline(-cfg.voltage_limit_v, linestyle="--", linewidth=0.8, alpha=0.4)
plt.xlabel("electrode center position (µm)")
plt.ylabel("solved voltage (V)")
plt.title("Electrode voltages for positioned wells")
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig(fig_dir / "well_positioning_voltages.png", dpi=180)
plt.show()

print(f"saved: {fig_dir / 'well_positioning_voltages.png'}")


## 6. Compact quality table

A first quality check: target location, achieved potential minimum, absolute error, voltage usage, and local curvature.


In [ ]:
rows = []
for s in solutions:
    rows.append([
        s["target_x"] * 1e6,
        s["min_x"] * 1e6,
        abs(s["min_x"] - s["target_x"]) * 1e6,
        np.max(np.abs(s["voltages"])),
        s["curvature"],
    ])

header = ["target_um", "min_um", "abs_error_um", "max_abs_voltage", "curvature"]
print(" | ".join(f"{h:>16s}" for h in header))
print("-" * 92)
for r in rows:
    print(f"{r[0]:16.2f} | {r[1]:16.2f} | {r[2]:16.2f} | {r[3]:16.3f} | {r[4]:16.3e}")


## 7. Save example voltage solutions

These arrays become useful for later notebooks and for checking reproducibility.


In [ ]:
data_dir = repo / "data" / "simulation_outputs"
data_dir.mkdir(parents=True, exist_ok=True)

np.savez(
    data_dir / "well_positioning_solutions.npz",
    x_grid=x,
    electrode_positions=electrode_positions,
    target_positions=targets,
    voltages=np.vstack([s["voltages"] for s in solutions]),
    potentials=np.vstack([s["potential"] for s in solutions]),
)

print(f"saved: {data_dir / 'well_positioning_solutions.npz'}")


## 8. Next notebook

`02_transport_path.ipynb` will convert fixed target positions into a continuous smooth path:

```text
well positions → smooth x_c(t) → shuttling path
```

That path then becomes the target input for waveform generation.
